In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1363")
catalog = dbutils.widgets.get("catalog")

In [0]:
import pyspark.sql.functions as F

df_stores_bronze = spark.read.table(f"{catalog}.bronze.stores")

df_stores_silver = (
    df_stores_bronze
        .withColumn('store_id', F.upper(F.nullif(F.trim(F.col('store_id')), F.lit(''))))
        .withColumn('store_name', F.trim(F.col('store_name')))
        .withColumn('city', F.initcap(F.trim(F.col('city'))))
        .withColumn('state', F.initcap(F.trim(F.col('state'))))
        .withColumn('region', F.initcap(F.trim(F.col('region'))))
        .drop('source_filename', 'ingested_at')
)

df_stores_silver.display()
df_stores_silver.write.mode('overwrite').saveAsTable(f"{catalog}.silver.stores")

## Performing SCD type 1 in the products

In [0]:
import pyspark.sql.functions as F

df_products_bronze = spark.read.table(f"{catalog}.bronze.products")

# category is a struct with l1 and l2, so split into two columns

df_products_silver_stg = (
    df_products_bronze
        .withColumn('product_id', F.upper(F.nullif(F.trim(F.col('product_id')), F.lit(''))))
        .withColumn('product_name', F.initcap(F.trim(F.col('product_name'))))
        .withColumn('brand', F.initcap(F.trim(F.col('brand'))))
        .withColumn('category_l1', F.col('category').getField('l1'))
        .withColumn('category_l2', F.col('category').getField('l2'))
        .withColumn('pack_size', F.col('pack').getField('size'))
        .withColumn('pack_unit', F.col('pack').getField('unit'))
        .select('product_id', 'product_name', 'brand', 'category_l1', 'category_l2', 'pack_size', 'pack_unit', 'mrp', 'selling_price', 'tags', 'snapshot_date')
)

df_products_silver_stg.display()

In [0]:
%sql
Create table if not exists identifier(:catalog).silver.products_type1 (
    product_id string,
    product_name string,
    brand string,
    category_l1 string,
    category_l2 string,
    pack_size int,
    pack_unit string,
    mrp decimal(10, 2),
    selling_price decimal(10, 2),
    is_active boolean,
    tags array<string>,
    last_changed_snapshot_date date
)

In [0]:
from delta.tables import DeltaTable
target = DeltaTable.forName(spark, f"{catalog}.silver.products_type1")

(target.alias('t')
    .merge(
        source=df_products_silver_stg.alias('s'),
        condition = 't.product_id = s.product_id'
    )
    .whenMatchedUpdate(set = {
        "product_name": "s.product_name",
        "brand": "s.brand",
        "category_l1": "s.category_l1",
        "category_l2": "s.category_l2",
        "pack_size": "s.pack_size",
        "pack_unit": "s.pack_unit",
        "mrp": "s.mrp",
        "selling_price": "s.selling_price",
        "tags": "s.tags",
        "last_changed_snapshot_date": "s.snapshot_date"
    })
    .whenNotMatchedInsert(values={
        "product_id": "s.product_id",
        "product_name": "s.product_name",
        "brand": "s.brand",
        "category_l1": "s.category_l1",
        "category_l2": "s.category_l2",
        "pack_size": "s.pack_size",
        "pack_unit": "s.pack_unit",
        "mrp": "s.mrp",
        "selling_price": "s.selling_price",
        "tags": "s.tags",
        "last_changed_snapshot_date": "s.snapshot_date"
    })
    .execute()
)

## Performing SCD type 2 on customers

In [0]:
df_customers_bronze = spark.read.table(f"{catalog}.bronze.customers")

df_customers_silver_stg = (
    df_customers_bronze
        .withColumn('customer_id', F.upper(F.nullif(F.trim(F.col('customer_id')), F.lit(''))))
        .withColumn('full_name', F.initcap(F.trim(F.col('full_name'))))
        .withColumn('email', F.lower(F.trim(F.col('email'))))
        .withColumn('phone', F.regexp_replace(F.trim(F.col('phone')), r'\D', '').substr(-10, 10))
        .withColumn('city', F.initcap(F.trim(F.col('city'))))
        .withColumn('tier', F.upper(F.trim(F.col('tier'))))
        .withColumn('signup_date', F.coalesce(F.try_to_date(F.col('signup_date'), 'yyyy-MM-dd'), F.try_to_date(F.col('signup_date'), 'dd/MM/yyyy')))
        .select('customer_id', 'full_name', 'email', 'phone', 'city', 'tier', 'signup_date')
)
df_customers_silver_stg.display()

In [0]:
%sql
create table if not exists identifier(:catalog).silver.customers_type2(
    customer_id string,
    full_name string,
    email string,
    phone string,
    city string,
    tier string,
    signup_date date,
    effective_from date,
    effective_to date,
    is_current boolean
)

In [0]:
from delta.tables import DeltaTable
target = DeltaTable.forName(spark, f"{catalog}.silver.customers_type2")

(
    target.alias('t')
    .merge(
        df_customers_silver_stg.alias('s'),
        "t.customer_id = s.customer_id AND t.is_current=true"
    )
    .whenMatchedUpdate(
        condition=(
            (F.col('t.full_name') != F.col('s.full_name')) |
            (F.col('t.email') != F.col('s.email')) |
            (F.col('t.phone') != F.col('s.phone')) |
            (F.col('t.city') != F.col('s.city')) |
            (F.col('t.tier') != F.col('s.tier')) |
            (F.col('t.signup_date') != F.col('s.signup_date'))
        ),
        set={
            "effective_to": F.current_date(),
            "is_current": F.lit(False)
        }
    )
    .whenNotMatchedInsert(values={
        "customer_id": F.col("s.customer_id"),
        "full_name": F.col("s.full_name"),
        "email": F.col("s.email"),
        "phone": F.col("s.phone"),
        "city": F.col("s.city"),
        "tier": F.col("s.tier"),
        "signup_date": F.col("s.signup_date"),
        "effective_from": F.current_date(),
        "effective_to": F.lit(None),
        "is_current": F.lit(True)
    })
    .execute()
)